# Field Scan : maladies v10 (v10b), Kaggle

Ce qui change par rapport à la v2 :
- **plus de sources** en apprentissage : en plus de la Tanzanie 2023 et du Pérou, BananaLSD (Bangladesh), cm8zf62kzh (Bangladesh 2026), BLADE (Inde) et godliver (Ouganda, fond blanc) ;
- **4 classes** : saine, black_sigatoka, fusarium, **autre** (insectes, Sigatoka jaune, Moko ou feuille morte, flétrissement bactérien, Cordana, Pestalotiopsis). L'alerte part sur 1 − P(saine) ;
- **seuils réglés sur une validation à plusieurs sources** (Tanzanie 2025 et 25 % de chaque nouvelle source, par groupes de copies), chaque source pesant autant.

Aucun juge ni test n'est lu ici. Les juges (khadavy, BDR) et le test neuf sont notés hors de Kaggle.
Réglages de la variante : `{'RESEAU': 'timm:convnext_tiny.dinov3_lvd1689m', 'TAILLE': 224, 'NB_MODELES': 3, 'EPOQUES': 6, 'TAILLE_LOT': 32, 'TAUX': 0.0001, 'TAUX_TETE': 0.001}`.


In [ ]:
# 1. GPU, emplacement des données
import os, sys, io, json, time, random
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import numpy as np, pandas as pd, torch
t448 = sorted(Path('/kaggle/input').rglob('images.csv'))
t10 = sorted(Path('/kaggle/input').rglob('images_v10.csv'))
assert t448 and t10, 'ajouter les jeux feuilles-448 et feuilles-v10'
RACINE, RACINE10 = t448[0].parent, t10[0].parent
print(RACINE, RACINE10)
appareil = 'cuda'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Réglages de la variante
NOM = 'v10b'
RESEAU = 'timm:convnext_tiny.dinov3_lvd1689m'
TAILLE = 224
NB_MODELES = 3
EPOQUES = 6
TAILLE_LOT = 32
TAUX = 0.0001
TAUX_TETE = 0.001
SANS_AUTRE = globals().get('SANS_AUTRE', False)
CLASSES = ['saine', 'black_sigatoka', 'fusarium'] + ([] if SANS_AUTRE else ['autre'])
POIDS_SOURCE = {'tanzanie_2023': 1.0, 'perou': 3.0, 'bananalsd': 2.0, 'cm8zf62kzh': 2.0, 'blade': 2.0, 'godliver': 2.0}
LISSAGE = 0.1
COUT_RATEE = 2              # rater une malade compte 2 fois une fausse alerte
OBJECTIF_NOM_JUSTE = 0.95
ACCEPTATION = 0.95          # « inconnu » : garder 95 % des photos de validation
GRAINE = 0
SORTIE = Path('/kaggle/working') / NOM
SORTIE.mkdir(parents=True, exist_ok=True)
random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)

In [ ]:
# 3. Apprentissage et validation (aucun juge, aucun test)
im = pd.read_csv(RACINE / 'images.csv')
def lire_liste(nom):
    return {l.strip().rsplit('.', 1)[0] for l in open(RACINE / nom, encoding='utf-8') if l.strip() and not l.startswith('#')}
courte = im.source.map(lambda c: '/'.join(c.split('/')[-2:]).rsplit('.', 1)[0])
retire = (im.jeu.str.startswith('tanzanie') & courte.isin(lire_liste('doublons_tanzanie.txt'))) | \
         ((im.jeu == 'bangladesh') & courte.isin(lire_liste('doublons_bangladesh.txt')))
im = im[~retire].copy()
im['jeu'] = im.jeu.replace({'bangladesh': 'bananalsd'})
im['classe'] = im.classe.replace({'sigatoka': 'black_sigatoka', 'cordana': 'autre', 'pestalotiopsis': 'autre'})
im['partie'] = np.where(im.jeu == 'tanzanie_2025', 'validation', 'apprentissage')
rng = np.random.RandomState(0)
for c in sorted(im[im.jeu == 'bananalsd'].classe.unique()):        # BananaLSD : 25 % en validation, par classe
    ids = im.index[(im.jeu == 'bananalsd') & (im.classe == c)].to_numpy().copy()
    rng.shuffle(ids)
    im.loc[ids[:round(len(ids) * 0.25)], 'partie'] = 'validation'
im['racine'] = str(RACINE)
v10 = pd.read_csv(RACINE10 / 'images_v10.csv')
v10['racine'] = str(RACINE10)
COLS = ['jeu', 'classe', 'image', 'partie', 'racine']
tout = pd.concat([im[COLS], v10[COLS]], ignore_index=True)
if SANS_AUTRE:
    tout = tout[tout.classe != 'autre']
assert set(tout.classe) <= set(CLASSES), set(tout.classe)
ent = tout[tout.partie == 'apprentissage'].reset_index(drop=True)
val = tout[tout.partie == 'validation'].reset_index(drop=True)
print(tout.groupby(['jeu', 'classe', 'partie']).size().unstack(fill_value=0).to_string())

In [ ]:
# 4. Réseau
import subprocess
if RESEAU.startswith('timm:'):        # poids DINOv3 et Pl@ntNet : timm récent
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm>=1.0.20', 'huggingface_hub'], check=False)
import timm
print('timm', timm.__version__)
def construire():
    if RESEAU == 'convnext_tiny':
        from torchvision import models
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = torch.nn.Linear(m.classifier[2].in_features, len(CLASSES))
        moy, ect = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        tete = list(m.classifier[2].parameters())
    else:
        nom_timm = RESEAU.split(':', 1)[1]
        try:
            m = timm.create_model(nom_timm, pretrained=True, num_classes=len(CLASSES), img_size=TAILLE, dynamic_img_size=True)
        except TypeError:
            m = timm.create_model(nom_timm, pretrained=True, num_classes=len(CLASSES))
        cfg = timm.data.resolve_data_config({}, model=m)
        moy, ect = cfg['mean'], cfg['std']
        tete = list(m.get_classifier().parameters())
    return m.to(appareil).to(memory_format=torch.channels_last), moy, ect, tete
_m, MOY, ECT, _ = construire()
print(RESEAU, f'{sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M paramètres ; moyenne {MOY} écart {ECT}')
del _m

In [ ]:
# 5. Images et augmentations (celles de la v2)
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
NORMALISATION = T.Normalize(MOY, ECT)
class JPEGAleatoire:
    def __init__(self, p=0.5, qualites=(30, 95)):
        self.p, self.qualites = p, qualites
    def __call__(self, image):
        if random.random() >= self.p:
            return image
        b = io.BytesIO(); image.save(b, format='JPEG', quality=random.randint(*self.qualites))
        return Image.open(b).convert('RGB')
aug_ent = T.Compose([JPEGAleatoire(),
    T.RandomResizedCrop(TAILLE, scale=(0.4, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.04),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    T.ToTensor(), NORMALISATION, T.RandomErasing(p=0.5, scale=(0.02, 0.15))])
def aug_eval(taille):
    return T.Compose([T.Resize(taille), T.ToTensor(), NORMALISATION])
class Feuilles(Dataset):
    def __init__(self, df, transformation):
        self.chemins = [Path(r) / i for r, i in zip(df.racine, df.image)]
        self.y = [CLASSES.index(c) for c in df.classe]
        self.t = transformation
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        return self.t(Image.open(self.chemins[i]).convert('RGB')), self.y[i]
poids = ent.jeu.map(POIDS_SOURCE).to_numpy()
assert not np.isnan(poids).any()
print('part de chaque source dans un tour :', (pd.Series(poids).groupby(ent.jeu).sum() / poids.sum()).round(3).to_dict())

In [ ]:
# 6. Entraînement (dernière époque gardée, comme la v2)
VUES = [lambda x: x, lambda x: x.flip(3), lambda x: x.flip(2), lambda x: x.rot90(1, (2, 3))]
def entrainer(graine):
    torch.manual_seed(graine); random.seed(graine); np.random.seed(graine)
    m, _, _, tete = construire()
    ids_tete = {id(p) for p in tete}
    corps = [p for p in m.parameters() if id(p) not in ids_tete]
    opt = torch.optim.AdamW([{'params': corps, 'lr': TAUX}, {'params': tete, 'lr': TAUX_TETE}], weight_decay=0.05 if RESEAU != 'convnext_tiny' else 1e-4)
    tirage = WeightedRandomSampler(torch.tensor(poids, dtype=torch.double), num_samples=len(ent), replacement=True)
    charge = DataLoader(Feuilles(ent, aug_ent), TAILLE_LOT, sampler=tirage, num_workers=4, pin_memory=True, drop_last=True)
    pas_total = EPOQUES * len(charge)
    echauffement = len(charge) // 2
    planning = torch.optim.lr_scheduler.LambdaLR(opt, lambda k: min(1.0, (k + 1) / echauffement) * 0.5 * (1 + np.cos(np.pi * min(k, pas_total) / pas_total)))
    perte = torch.nn.CrossEntropyLoss(label_smoothing=LISSAGE)
    echelle = torch.cuda.amp.GradScaler()
    for e in range(1, EPOQUES + 1):
        m.train(); debut, total, n = time.time(), 0.0, 0
        for x, y in charge:
            x, y = x.to(appareil, non_blocking=True).to(memory_format=torch.channels_last), y.to(appareil)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                l = perte(m(x).float(), y)
            echelle.scale(l).backward(); echelle.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            echelle.step(opt); echelle.update(); planning.step()
            total += l.item() * len(y); n += len(y)
        assert np.isfinite(total), 'perte NaN'
        print(f'graine {graine} époque {e} perte {total / n:.3f} ({time.time() - debut:.0f} s)', flush=True)
    m.eval()
    torch.save({k: v.half() if v.is_floating_point() else v for k, v in m.state_dict().items()}, SORTIE / f'maladies_{NOM}_{graine}.pt')
    return m
@torch.no_grad()
def probas(m, df, taille):
    charge = DataLoader(Feuilles(df, aug_eval(taille)), 64, num_workers=4)
    out = []
    for x, _ in charge:
        x = x.to(appareil).to(memory_format=torch.channels_last); p = 0
        for v in VUES:
            with torch.autocast('cuda', dtype=torch.float16):
                p = p + torch.softmax(m(v(x)).float(), 1)
        out.append((p / len(VUES)).cpu())
    return torch.cat(out).numpy()
P = {t: [] for t in (TAILLE, 392 if 'patch14' in RESEAU else 384)}   # patch de 14 px : 392 au lieu de 384
debut = time.time()
for g in range(NB_MODELES):
    m = entrainer(g)
    for t in P:
        P[t].append(probas(m, val, t))
    del m; torch.cuda.empty_cache()
    print(f'réseau {g} fini ({(time.time() - debut) / 60:.0f} min)', flush=True)
P = {t: np.mean(v, 0) for t, v in P.items()}
for t, p in P.items():
    d = val[['jeu', 'classe', 'image']].copy()
    for k, c in enumerate(CLASSES):
        d[f'p_{c}'] = p[:, k]
    d.to_csv(SORTIE / f'probas_validation_{t}.csv', index=False)

In [ ]:
# 7. Seuils réglés sur la validation (chaque source pèse autant), décision et bilan par source
y = val.classe.map(CLASSES.index).to_numpy()
src = val.jeu.to_numpy()
SOURCES = sorted(set(src))
def regler(p):
    pm = 1 - p[:, 0]
    def cout(s):
        c = []
        for j in SOURCES:
            k = src == j; mal = y[k] > 0; dit = pm[k] >= s
            c.append(COUT_RATEE * (dit[mal] == False).mean() + (dit[~mal].mean() if (~mal).any() else 0))
        return np.mean(c)
    grille = np.round(np.arange(0.05, 0.951, 0.025), 3)
    sa = float(grille[int(np.argmin([cout(s) for s in grille]))])
    part = p[:, 1:].max(1) / np.maximum(pm, 1e-9); k = 1 + p[:, 1:].argmax(1)
    alerte = (pm >= sa) & (y > 0)
    sn = 1.0
    for s in np.round(np.arange(0.50, 1.0, 0.01), 2):
        nom = alerte & (part >= s) & (k < 3)
        if nom.sum() and (k[nom] == y[nom]).mean() >= OBJECTIF_NOM_JUSTE:
            sn = float(s); break
    sc = float(np.quantile(p.max(1), 1 - ACCEPTATION))
    return {'seuil_alerte': sa, 'seuil_nom': sn, 'seuil_confiance': sc}
def decider(p, s):
    pm = 1 - p[:, 0]; part = p[:, 1:].max(1) / np.maximum(pm, 1e-9)
    nom = np.array(CLASSES)[1 + p[:, 1:].argmax(1)]
    r = np.where(pm < s['seuil_alerte'], 'saine', np.where((nom != 'autre') & (part >= s['seuil_nom']), nom, 'malade_non_identifiee')).astype(object)
    r[p.max(1) < s['seuil_confiance']] = 'inconnu'
    return r
def bilan(r):
    lignes = []
    for j in SOURCES:
        k = src == j; v = val.classe.to_numpy()[k]; rr = r[k]; mal = v != 'saine'
        nomme = np.isin(rr, ['black_sigatoka', 'fusarium'])
        lignes.append({'source': j, 'malades_dites_saines': float((rr[mal] == 'saine').mean()) if mal.any() else None,
                       'fausses_alertes': float(np.isin(rr[~mal], ['black_sigatoka', 'fusarium', 'malade_non_identifiee']).mean()) if (~mal).any() else None,
                       'saines_inconnu': float((rr[~mal] == 'inconnu').mean()) if (~mal).any() else None,
                       'nom_juste': float((rr[nomme] == v[nomme]).mean()) if nomme.any() else None, 'n': int(k.sum())})
    return pd.DataFrame(lignes)
resultats = {}
for t, p in P.items():
    s = regler(p)
    b = bilan(decider(p, s))
    print(f'--- vu en {t} px : {s}'); print(b.round(3).to_string(index=False))
    json.dump({'modele': NOM, 'reseau': RESEAU, 'classes': CLASSES, 'taille_apprentissage': TAILLE, 'taille': t,
               'nb_modeles': NB_MODELES, 'quatre_vues': True, 'moyenne': list(MOY), 'ecart': list(ECT), **s,
               'decision': "alerte si 1 - P(saine) >= seuil_alerte ; maladie nommée si la plus probable des maladies n'est pas « autre » et fait au moins seuil_nom de 1 - P(saine), sinon malade_non_identifiee ; inconnu si la plus forte probabilité est sous seuil_confiance",
               'note': 'v10 : seuils réglés sur la validation Tanzanie 2025 + 25 % de BananaLSD, cm8zf62kzh, BLADE, godliver (chaque source pèse autant).'},
              open(SORTIE / f'seuils_{t}.json', 'w'), indent=1, ensure_ascii=False)
    resultats[t] = {'seuils': s, 'bilan': b.to_dict('records')}
json.dump(resultats, open(SORTIE / 'resultats.json', 'w'), indent=1, ensure_ascii=False)
print(sorted(p.name for p in SORTIE.iterdir()))